# CLRS red-black tree, with selectors written in Python

The [`spytial-clrs`](https://github.com/sidprasad/spytial-clrs) notebooks declare
their spatial constraints with **sgq**, the graph query language spytial-core
evaluates in the browser. sgq runs over the *relationalized* instance, so a
selector has to be written in terms of what the relationalizer emits rather than
in terms of the objects. In `trees.ipynb`, skipping the NIL sentinel reads:

```
left & (RBNode -> (RBNode - NoneType.~key))
```

A `selector` may instead be a **Python function**. It runs during
`spytial.diagram()`, after the walk and before the specification is written, so
it can be handed the values the walk reached and return the ones to select.
spytial translates each returned value to the ID of its atom.

This notebook is CLRS chapter 13 written that way. The structure and the
resulting diagram are the same; only the selectors differ.

In [1]:
import spytial
from spytial import BorderStyle

## The selectors

Each function receives `values` -- everything the walk reached, which is exactly
the set of values that have atoms -- and returns rows: one value for a unary
selector, a tuple for a binary one.

The functions may name `RBNode` and `NIL` even though neither exists yet. They
are not called until diagram time.

In [2]:
RED, BLACK = "red", "black"


# --- selectors -------------------------------------------------------------
# Each runs at diagram time, so it may name RBNode and NIL, which do not exist
# while the decorators below are being applied.

def real(n):
    """A node that is not the NIL sentinel."""
    return isinstance(n, RBNode) and n is not NIL


def left_edges(values):
    return [(n, n.left) for n in values if real(n) and real(n.left)]


def right_edges(values):
    return [(n, n.right) for n in values if real(n) and real(n.right)]


def red_nodes(values):
    return [n for n in values if real(n) and n.color == RED]


def black_nodes(values):
    return [n for n in values if real(n) and n.color == BLACK]


def scaffolding(values):
    """Everything the diagram should not draw: the NIL sentinel, the tree
    wrapper, and the raw ints and colour strings that `attribute` already
    prints inside each node."""
    return [
        v for v in values
        if v is NIL or isinstance(v, (RBTree, int)) or v is None or v in (RED, BLACK)
    ]

## The node

The decorators are the ordinary ones. Only the `selector` arguments changed.

In [3]:
@spytial.orientation(selector=left_edges, directions=["below", "left"])
@spytial.orientation(selector=right_edges, directions=["below", "right"])
@spytial.atomStyle(selector=red_nodes, borderStyle=BorderStyle(color="red"))
@spytial.atomStyle(selector=black_nodes, borderStyle=BorderStyle(color="black"))
@spytial.hideAtom(selector=scaffolding)
@spytial.hideField(field="parent")
@spytial.attribute(field="key")
@spytial.attribute(field="color")
class RBNode:
    def __init__(self, key=None, color=BLACK, left=None, right=None, parent=None):
        self.key, self.color = key, color
        self.left, self.right, self.parent = left, right, parent


NIL = RBNode(key=None, color=BLACK)
NIL.left = NIL.right = NIL.parent = NIL

## RB-INSERT and RB-INSERT-FIXUP (CLRS 13.3)

In [4]:
class RBTree:
    """CLRS 13.3: RB-INSERT and RB-INSERT-FIXUP."""

    def __init__(self):
        self.root = NIL

    def left_rotate(self, x):
        y = x.right
        x.right = y.left
        if y.left is not NIL:
            y.left.parent = x
        y.parent = x.parent
        if x.parent is NIL:
            self.root = y
        elif x is x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left, x.parent = x, y

    def right_rotate(self, y):
        x = y.left
        y.left = x.right
        if x.right is not NIL:
            x.right.parent = y
        x.parent = y.parent
        if y.parent is NIL:
            self.root = x
        elif y is y.parent.left:
            y.parent.left = x
        else:
            y.parent.right = x
        x.right, y.parent = y, x

    def insert(self, key):
        z = RBNode(key=key, color=RED, left=NIL, right=NIL, parent=NIL)
        y, x = NIL, self.root
        while x is not NIL:
            y = x
            x = x.left if z.key < x.key else x.right
        z.parent = y
        if y is NIL:
            self.root = z
        elif z.key < y.key:
            y.left = z
        else:
            y.right = z
        self._fixup(z)

    def _fixup(self, z):
        while z.parent.color == RED:
            if z.parent is z.parent.parent.left:
                y = z.parent.parent.right
                if y.color == RED:
                    z.parent.color = y.color = BLACK
                    z.parent.parent.color = RED
                    z = z.parent.parent
                else:
                    if z is z.parent.right:
                        z = z.parent
                        self.left_rotate(z)
                    z.parent.color = BLACK
                    z.parent.parent.color = RED
                    self.right_rotate(z.parent.parent)
            else:
                y = z.parent.parent.left
                if y.color == RED:
                    z.parent.color = y.color = BLACK
                    z.parent.parent.color = RED
                    z = z.parent.parent
                else:
                    if z is z.parent.left:
                        z = z.parent
                        self.right_rotate(z)
                    z.parent.color = BLACK
                    z.parent.parent.color = RED
                    self.left_rotate(z.parent.parent)
        self.root.color = BLACK

## The tree from CLRS figure 13.4

In [5]:
t = RBTree()
for k in [41, 38, 31, 12, 19, 8]:
    t.insert(k)

spytial.diagram(t)

## What the two spellings look like side by side

| Constraint | sgq (`trees.ipynb`) | Python (this notebook) |
| --- | --- | --- |
| Left edges, skipping NIL | `left & (RBNode -> (RBNode - NoneType.~key))` | `[(n, n.left) for n in values if real(n) and real(n.left)]` |
| Red nodes | `{ x : RBNode \| @:(x.color) = "red" }` | `[n for n in values if real(n) and n.color == RED]` |
| Scaffolding to hide | `{ x : RBNode \| (x.key in NoneType) } + RBTree + int + NoneType + {s : str \| @:s = "red" or @:s = "black"}` | `[v for v in values if v is NIL or isinstance(v, (RBTree, int)) or v is None or v in (RED, BLACK)]` |

Two differences are worth naming.

The NIL sentinel is a single object, so Python identifies it with `n is NIL`.
sgq has no way to say "this object", so `trees.ipynb` identifies it by a
property that happens to hold only of it -- its key is `None`, hence
`NoneType.~key`. That works, and it stops working the moment a real node is
given a `None` key.

The Python form uses the object's own attributes. `n.color == RED` is the same
comparison the algorithm makes, so the selector and the code it illustrates
cannot drift apart.

## Where sgq is still the better spelling

Python selectors are not a replacement. `disjoint-sets.ipynb` orients a DSU
forest with:

```
^(~parent)
```

That is the transitive closure of the inverted `parent` relation -- every
ancestor edge, at any depth. There is no comprehension for it; the Python
version would be a hand-written fixpoint loop, which is longer and easier to get
wrong. Closure, transposition, and the other relational operators are what sgq
is for.

A rough rule: reach for a Python selector when the condition is about a
**value** (`n.color == RED`, `n is NIL`, `len(n.keys) > 2`), and for sgq when it
is about the **shape of the graph** (`^parent`, `~next`, `iden`).

## Two limits

A translated selector names atom IDs, and an atom ID is a position in one walk.
After an insertion the same ID denotes a different node, so a Python selector
describes exactly the instance it was translated against. `spytial.sequence()`
and `spytial.edit()` render several instances from one specification and reject
one with a `SelectorError`; write those as sgq.

A value the walk never reached cannot be translated. It is dropped with an
`AtomNotInInstance` warning naming it, and the rest of the selector still
applies.